In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile, shutil
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
m=list(INPUT.rglob('active_tb_safe_eval2_v1/EVAL_BOUNDARY.json'))
if len(m)==1: SRC=m[0].parent
else:
 a=list(INPUT.rglob('active_tb_safe_eval2_v1.zip')); assert len(a)==1,a
 d=WORK/'att'
 with zipfile.ZipFile(a[0]) as zf: zf.extractall(d)
 SRC=d/'active_tb_safe_eval2_v1'
print(json.loads((SRC/'EVAL_BOUNDARY.json').read_text()))
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','iverilog'],check=True)


In [ ]:
OUT=WORK/'eval2'; OUT.mkdir(exist_ok=True)
for fe in ('correctbench','autobench'):
    d=SRC/fe
    sz=d/'suites'
    if not sz.is_dir():
        sz=WORK/f'suites_{fe}'
        with zipfile.ZipFile(d/'suites.zip') as zf: zf.extractall(sz)
    print('='*25, fe, '='*25, flush=True)
    r=subprocess.run([sys.executable,str(SRC/'sealed_dual_eval.py'),
      '--suites',str(sz),'--manifest',str(d/'safe_manifest.json'),
      '--checkers',str(d/'checkers.jsonl'),'--golden',str(SRC/'golden.jsonl'),
      '--mutants',str(SRC/'hidden_bank_a.jsonl'),
      '--out',str(OUT/f'{fe}_records.csv')],text=True,capture_output=True)
    print(r.stdout[-3000:])
    if r.returncode: print('STDERR:',r.stderr[-2000:])
shutil.make_archive(str(WORK/'eval2'),'zip',OUT)
print('packaged')
